In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

df = pd.read_csv("fake_stock.csv")
gold = df[df["metal"] == "Gold"].copy()

PURITY_FACTORS = {"24K": 0.999, "22K": 0.916, "18K": 0.75, "14K": 0.585}
gold["purity_factor"] = gold["purity"].map(PURITY_FACTORS)

gold[["weight_gm", "purity", "purity_factor"]].head()
np.random.seed(42)
noise = np.random.uniform(-1, 1, len(gold))
gold["price_k"] = gold["weight_gm"] * gold["purity_factor"] * 7.1 + 2 + noise

gold[["weight_gm", "purity", "price_k"]].head()

,weight_gm,purity,purity_factor
0,16.20,22K,0.916
1,26.47,22K,0.916
2,13.53,22K,0.916
3,46.01,24K,0.999
4,57.52,18K,0.750


In [ ]:
np.random.seed(42)
noise = np.random.uniform(-1, 1, len(gold))
gold["price_k"] = gold["weight_gm"] * gold["purity_factor"] * 7.1 + 2 + noise

gold[["weight_gm", "purity", "price_k"]].head()

,weight_gm,purity,price_k
0,16.20,22K,107.107400
1,26.47,22K,175.051721
2,13.53,22K,90.457696
3,46.01,24K,328.541646
4,57.52,18K,307.606037


In [5]:
gold["weight_scaled"] = gold["weight_gm"] / 60

train = gold.sample(frac=0.8, random_state=42)
test = gold.drop(train.index)

cols = ["weight_scaled", "purity_factor"]
x_train = torch.tensor(train[cols].values, dtype=torch.float32)
y_train = torch.tensor(train[["price_k"]].values, dtype=torch.float32)
x_test  = torch.tensor(test[cols].values, dtype=torch.float32)
y_test  = torch.tensor(test[["price_k"]].values, dtype=torch.float32)

print(x_train.shape)
print(x_train[:3])

torch.Size([3190, 2])
tensor([[0.8893, 0.9160],
        [0.8030, 0.7500],
        [0.9372, 0.9160]])


In [ ]:
y_scale = 100

y_train_s = y_train / y_scale     # prices now ~0.15 to ~4.3
y_test_s  = y_test / y_scale
x_mean = x_train.mean(dim=0)       # average of each column (training data only)
x_std  = x_train.std(dim=0)        # spread of each column

x_train_n = (x_train - x_mean) / x_std
x_test_n  = (x_test  - x_mean) / x_std     # SAME mean and std (no leakage)

print(x_train_n[:5])
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(5000):
    pred = model(x_train_n)                 # ← standardised inputs
    loss = loss_fn(pred, y_train_s)         # ← prices still ÷ 100
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(f"step {step:4d}  loss = {loss.item():.5f}")

with torch.no_grad():
    print("training loss (real ₹k):", loss_fn(model(x_train_n) * y_scale, y_train).item())
    print("test loss (real ₹k):    ", loss_fn(model(x_test_n) * y_scale, y_test).item())




tensor([[ 1.3596,  0.6736],
        [ 1.0480, -0.3702],
        [ 1.5322,  0.6736],
        [ 0.1771,  1.1955],
        [ 0.8345,  1.1955]])
step    0  loss = 5.32173
step 1000  loss = 0.00004
step 2000  loss = 0.00004
step 3000  loss = 0.00003
step 4000  loss = 0.00003
training loss (real ₹k): 0.3602263629436493
test loss (real ₹k):     0.3725129961967468


In [19]:
items = torch.tensor([
    [10/60, 0.999],
    [10/60, 0.585],
    [50/60, 0.999],
    [50/60, 0.585],
])
items_n = (items - x_mean) / x_std

with torch.no_grad():
    print(model(items_n) * y_scale)     # remember: × y_scale to get real prices

tensor([[ 73.2345],
        [ 43.8169],
        [356.6995],
        [209.8749]])


In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
class PriceModel(nn.Module):
    
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden = nn.Linear(2, hidden_size)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_size,1)
    def forward(self,x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x
y_scale = 100
df = pd.read_csv("fake_stock.csv")
gold = df[df["metal"] == "Gold"].copy()

PURITY_FACTORS = {"24K": 0.999, "22K": 0.916, "18K": 0.75, "14K": 0.585}
gold["purity_factor"] = gold["purity"].map(PURITY_FACTORS)

gold[["weight_gm", "purity", "purity_factor"]].head()
np.random.seed(42)
noise = np.random.uniform(-1, 1, len(gold))
gold["price_k"] = gold["weight_gm"] * gold["purity_factor"] * 7.1 + 2 + noise

gold[["weight_gm", "purity", "price_k"]].head()
gold["weight_scaled"] = gold["weight_gm"] / 60

train = gold.sample(frac=0.8, random_state=42)
test = gold.drop(train.index)

cols = ["weight_scaled", "purity_factor"]
x_train = torch.tensor(train[cols].values, dtype=torch.float32)
y_train = torch.tensor(train[["price_k"]].values, dtype=torch.float32)
x_test  = torch.tensor(test[cols].values, dtype=torch.float32)
y_test  = torch.tensor(test[["price_k"]].values, dtype=torch.float32)
y_train_s = y_train / y_scale     # prices now ~0.15 to ~4.3
y_test_s  = y_test / y_scale
x_mean = x_train.mean(dim=0)       # average of each column (training data only)
x_std  = x_train.std(dim=0)        # spread of each column

x_train_n = (x_train - x_mean) / x_std
x_test_n  = (x_test  - x_mean) / x_std     # SAME mean and std (no leakage)
for size in [4, 16, 64, 256]:
    torch.manual_seed(0)
    model = PriceModel(size)
    torch.manual_seed(0)
    loss_fn=nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for r in range(5000):
        pred = model(x_train_n)
        loss=loss_fn(pred,y_train_s)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if r % 1000 == 0:
            print(f"  step {r}  loss = {loss.item():.5f}")
        with torch.no_grad():
            test_pred = model(x_test_n) * y_scale
            test_loss = loss_fn(test_pred, y_test).item()
    print(size, "neurons → test loss:", round(test_loss, 3))




  step 0  loss = 4.08249
  step 1000  loss = 0.00343
  step 2000  loss = 0.00258
  step 3000  loss = 0.00205
  step 4000  loss = 0.00169
4 neurons → test loss: 14.0
  step 0  loss = 3.86127
  step 1000  loss = 0.00029
  step 2000  loss = 0.00015
  step 3000  loss = 0.00010
  step 4000  loss = 0.00008
16 neurons → test loss: 0.685
  step 0  loss = 5.32173
  step 1000  loss = 0.00004
  step 2000  loss = 0.00004
  step 3000  loss = 0.00003
  step 4000  loss = 0.00003
64 neurons → test loss: 0.373
  step 0  loss = 4.44120
  step 1000  loss = 0.00004
  step 2000  loss = 0.00003
  step 3000  loss = 0.00003
  step 4000  loss = 0.00003
256 neurons → test loss: 0.334
